# Módulo 4 · Notebook 4 — RAG Avanzado

Técnicas para mejorar la calidad del retrieval más allá del `similarity_search` básico.

| Técnica | Problema que resuelve |
|---------|----------------------|
| **MMR** | Chunks redundantes o muy parecidos |
| **Multi-query** | La pregunta del usuario no coincide con el texto indexado |
| **Filtro por metadatos** | Buscar solo en un documento o página |
| **Ajuste de chunk_size** | Chunks demasiado grandes o pequeños |

## 1. Setup

In [ ]:
import warnings
from pathlib import Path

from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

warnings.filterwarnings("ignore")

BASE_DIR = Path("../../").resolve()
VECTOR_DIR = str(BASE_DIR / "data" / "vector_db")

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vectorstore = Chroma(
    persist_directory=VECTOR_DIR,
    embedding_function=embeddings,
    collection_name="ml_papers",
)
llm = ChatOllama(model="llama3.2:1b", temperature=0)
query = "¿Cómo se entrenan modelos de lenguaje en el dominio financiero?"

print("✅ Vector store cargado")

## 2. Baseline — búsqueda por similitud

In [ ]:
baseline = vectorstore.similarity_search(query, k=3)
print(f"Baseline: {len(baseline)} chunks\n")
for i, d in enumerate(baseline, 1):
  print(f"[{i}] {Path(d.metadata.get('source','?')).name} p.{d.metadata.get('page','?')}")
  print(d.page_content[:200].replace("\n", " "), "...\n")

## 3. MMR — diversidad en los resultados

**Maximal Marginal Relevance:** equilibra relevancia y diversidad para evitar chunks casi idénticos.

In [ ]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.5},
)
mmr_docs = mmr_retriever.invoke(query)
print(f"MMR: {len(mmr_docs)} chunks\n")
for i, d in enumerate(mmr_docs, 1):
    print(f"[{i}] {Path(d.metadata.get('source','?')).name} p.{d.metadata.get('page','?')}")
    print(d.page_content[:200].replace("\n", " "), "...\n")

## 4. Multi-query — reformular la pregunta

El LLM genera varias versiones de la consulta y fusiona los resultados.

In [ ]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
)
mq_docs = multi_retriever.invoke(query)
print(f"Multi-query: {len(mq_docs)} chunks\n")
for i, d in enumerate(mq_docs, 1):
    print(f"[{i}] {Path(d.metadata.get('source','?')).name} p.{d.metadata.get('page','?')}")

## 5. Filtro por metadatos

Restringe la búsqueda a un paper concreto (`paper1.pdf` = FinGPT).

In [ ]:
paper1_path = str(BASE_DIR / "data" / "Papers" / "paper1.pdf")
filtered = vectorstore.similarity_search(
    query,
    k=3,
    filter={"source": paper1_path},
)
print(f"Filtrado a paper1.pdf: {len(filtered)} chunks\n")
for i, d in enumerate(filtered, 1):
    print(f"[{i}] p.{d.metadata.get('page','?')}: {d.page_content[:150]}...")

## 6. Impacto del tamaño de chunk

Compara cuántos caracteres recuperas con distintos valores de `k`.

In [ ]:
for k in [2, 5, 8]:
    docs = vectorstore.similarity_search(query, k=k)
    chars = sum(len(d.page_content) for d in docs)
    print(f"k={k} → {len(docs)} chunks, {chars:,} caracteres de contexto")

## 7. Resumen

- Usa **MMR** cuando los chunks se repiten.
- Usa **multi-query** cuando el usuario pregunta de forma vaga.
- Usa **filtros** cuando conoces el documento o sección relevante.
- Ajusta **k** según el presupuesto de tokens del LLM.

Siguiente: `05_eval_rag.ipynb` — cómo medir si el RAG funciona bien.